# D2-06 Variability and uncertainty in regionalised LCIA

Day 1 focused on uncertainty in inventories and exchange amounts. This notebook asks a different question: **what happens when the characterization factors (CFs) are uncertain or spatially variable?**

We start with two small teaching methods, then repeat the workflow with the regionalised `AWARE 2.0` method.

## Learning goals

After this notebook, you should be able to:

- distinguish inventory uncertainty from CF uncertainty
- read uncertainty information from an `edges` JSON method
- run deterministic and stochastic `EdgeLCIA` calculations
- interpret direct contributions, upstream contributions, quantiles, and coefficients of variation
- explain how AWARE balances water withdrawal and water return
- **Optional:** combine inventory and CF uncertainty in one calculation

## Course route

During the live course, complete **sections 1–7**. Section 8 is an optional extension.

The calculation pattern used throughout is:

1. inspect the method and inventory;
2. calculate a deterministic reference score;
3. sample uncertain CFs;
4. summarize and visualize the resulting score distribution.

## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087–3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891–907. https://doi.org/10.1111/jiec.70023

## 1) What changes relative to Day 1?

- **Inventory uncertainty:** the amount of an exchange varies, for example the kilograms of fuel consumed.
- **CF uncertainty:** the impact per unit of an exchange varies, for example the water-scarcity CF for a particular location.

In sections 2–7, the inventory is held fixed while CFs vary. Section 8 optionally varies both.

## 2) Inspect two small methods with uncertain CFs

The bundled JSON files are deliberately small:

- `lcia_example_5.json` uses continuous probability distributions for greenhouse-gas CFs;
- `lcia_example_6.json` uses a nested discrete/continuous distribution for gross water withdrawal.

First import the packages and define the two file paths.

In [ ]:
import json
import warnings
from pathlib import Path

import bw2data as bd
import country_converter as coco
import edges
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from edges import EdgeLCIA
from edges.uncertainty import sample_cf_distribution


example_5_path = Path("assets/edges_examples/lcia_example_5.json")
example_6_path = Path("assets/edges_examples/lcia_example_6.json")

Load each JSON file as a Python dictionary. The overview confirms the method names, units, and number of characterization rules.

In [ ]:
method_paths = {
    "GWP with uncertainty": example_5_path,
    "Water withdrawal with uncertainty": example_6_path,
}

method_data = {
    label: json.loads(path.read_text())
    for label, path in method_paths.items()
}
method_units = {
    label: data["unit"]
    for label, data in method_data.items()
}

method_overview = pd.DataFrame([
    {
        "method": label,
        "unit": data["unit"],
        "characterization rules": len(data["exchanges"]),
    }
    for label, data in method_data.items()
]).set_index("method")

method_overview

Each characterization rule has three important parts:

- **flow pattern:** which biosphere-flow names can match;
- **compartment:** where the flow occurs;
- **uncertainty distribution:** how the nominal CF can vary.

An empty compartment below means that the rule can match the named flow in any compartment.

In [ ]:
rule_rows = []

for method_label, data in method_data.items():
    for rule in data["exchanges"]:
        supplier = rule["supplier"]
        uncertainty = rule.get("uncertainty", {})
        rule_rows.append({
            "method": method_label,
            "flow pattern": supplier["name"],
            "compartment": " :: ".join(supplier.get("categories", [])),
            "nominal CF": rule["value"],
            "distribution": uncertainty.get("distribution", "none"),
        })

simple_cf_rules = pd.DataFrame(rule_rows)
simple_cf_rules

## 3) Select and inspect the functional-unit activity

We use the BAFU activity `Irrigating` in Switzerland. It contains direct fossil emissions and a direct river-water withdrawal, so both teaching methods have something to characterize.

Match the activity by **name, reference product, and location**, and fail clearly if the result is not unique.

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")
database = bd.Database("bafu")

activity_matches = [
    candidate
    for candidate in database
    if candidate["name"] == "Irrigating"
    and candidate["reference product"] == "Irrigating"
    and candidate.get("location") == "CH"
]

if len(activity_matches) != 1:
    raise ValueError(f"Expected one Irrigating activity; found {len(activity_matches)}")

activity = activity_matches[0]

pd.Series({
    "name": activity["name"],
    "reference product": activity["reference product"],
    "location": activity["location"],
    "functional unit": f'1 {activity["unit"]}',
}, name="Selected activity")

Now inspect only the direct flows relevant to the two teaching methods. A loop is used instead of a dense list comprehension so each filtering decision is visible.

In [ ]:
fossil_flows = {
    "Carbon dioxide, fossil",
    "Methane, fossil",
    "Nitrogen monoxide",
}
flow_rows = []

for exchange in activity.biosphere():
    flow = exchange.input
    name = flow["name"]
    categories = tuple(flow.get("categories", ()))

    if name in fossil_flows:
        role = "air emission"
    elif name.startswith("Water") and categories[:2] == ("natural resource", "in water"):
        role = "withdrawal"
    elif name == "Water" and categories == ("water",):
        role = "return to water"
    else:
        continue

    flow_rows.append({
        "role": role,
        "flow": name,
        "compartment": " :: ".join(categories),
        "amount": exchange["amount"],
        "unit": flow["unit"],
    })

relevant_biosphere_flows = pd.DataFrame(flow_rows)
relevant_biosphere_flows

The simple water method scores **gross withdrawal**. The inventory also contains a water return, but that return is not part of this teaching method.

The difference between withdrawal and return is the activity's implicit consumptive use; it is not explicitly labelled as evaporation.

In [ ]:
gross_withdrawal = relevant_biosphere_flows.loc[
    relevant_biosphere_flows["role"] == "withdrawal", "amount"
].sum()
water_return = relevant_biosphere_flows.loc[
    relevant_biosphere_flows["role"] == "return to water", "amount"
].sum()

pd.Series({
    "gross withdrawal (m³)": gross_withdrawal,
    "return to water (m³)": water_return,
    "implicit consumptive use (m³)": gross_withdrawal - water_return,
}, name="Direct water balance")

## 4) Establish deterministic reference scores

Before sampling uncertainty, calculate one score using each nominal CF. This gives us a reference point for the Monte Carlo results.

The helper below contains the four essential `edges` steps:

1. build the life-cycle inventory;
2. match exchanges to CF rules;
3. evaluate the CFs;
4. calculate the LCIA score.

In [ ]:
def run_deterministic_method(label, method_path):
    lca = EdgeLCIA(
        demand={activity: 1},
        method=("teaching", label),
        filepath=str(method_path),
    )
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()
    return lca

The total LCIA score includes the complete supply chain. To understand the foreground activity, we also need the characterized rows whose consumer is exactly `Irrigating (CH)`.

The next helper applies that same direct-row filter throughout the notebook.

In [ ]:
def get_direct_cf_rows(lca, consumer, cf_column="CF"):
    table = lca.generate_cf_table(include_unmatched=False)

    same_consumer = (
        (table["consumer name"] == consumer["name"])
        & (table["consumer reference product"] == consumer["reference product"])
        & (table["consumer location"] == consumer["location"])
    )
    has_cf = table[cf_column].notna()

    return table.loc[same_consumer & has_cf].copy()

Run both methods and keep the LCA object together with its directly matched rows.

In [ ]:
deterministic_results = {}

for label, method_path in method_paths.items():
    lca = run_deterministic_method(label, method_path)
    direct_rows = get_direct_cf_rows(lca, activity)

    deterministic_results[label] = {
        "lca": lca,
        "direct rows": direct_rows,
    }

The summary separates the direct contribution from the upstream remainder:

- **direct contribution:** characterized exchanges consumed by `Irrigating (CH)`;
- **upstream contribution:** everything else in the supply chain;
- **total score:** direct plus upstream contributions.

In [ ]:
deterministic_rows = []

for label, result in deterministic_results.items():
    direct_rows = result["direct rows"]
    total_score = float(result["lca"].score)
    direct_score = direct_rows["impact"].sum()

    deterministic_rows.append({
        "method": label,
        "unit": method_units[label],
        "direct matches": len(direct_rows),
        "direct contribution": direct_score,
        "upstream contribution": total_score - direct_score,
        "total score": total_score,
    })

deterministic_summary = pd.DataFrame(deterministic_rows).set_index("method")
deterministic_summary

**Interpretation:** one direct match does not mean that only one exchange contributes to the total. For water withdrawal, the 1,200 m³ foreground withdrawal is one match, while many additional withdrawals occur upstream.

## 5) Add CF uncertainty

The demand and inventory stay fixed. Only the CFs are sampled.

`use_distributions=True` activates the uncertainty blocks from the JSON files, and `iterations` sets the number of repeated LCIA calculations.

In [ ]:
def run_cf_monte_carlo(label, method_path, iterations):
    lca = EdgeLCIA(
        demand={activity: 1},
        method=("teaching", label),
        filepath=str(method_path),
        use_distributions=True,
        iterations=iterations,
    )
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()
    return lca

Run the same two methods for 1,000 iterations and store each score distribution as a NumPy array.

In [ ]:
iterations = 1_000
stochastic_results = {}

for label, method_path in method_paths.items():
    lca = run_cf_monte_carlo(label, method_path, iterations)
    stochastic_results[label] = {
        "lca": lca,
        "scores": np.asarray(lca.score, dtype=float),
    }

Summarize each distribution with:

- the **mean** and **median** (`p50`);
- the central 90% interval (`p05` to `p95`);
- the **coefficient of variation** (standard deviation divided by the mean), which allows a relative comparison across methods with different units.

In [ ]:
def summarize_scores(scores):
    scores = np.asarray(scores, dtype=float)
    return {
        "mean": scores.mean(),
        "p05": np.quantile(scores, 0.05),
        "p50": np.quantile(scores, 0.50),
        "p95": np.quantile(scores, 0.95),
        "coefficient of variation": scores.std(ddof=0) / abs(scores.mean()),
    }

In [ ]:
stochastic_rows = []

for label, result in stochastic_results.items():
    stochastic_rows.append({
        "method": label,
        "unit": method_units[label],
        "deterministic score": deterministic_results[label]["lca"].score,
        **summarize_scores(result["scores"]),
    })

stochastic_summary = pd.DataFrame(stochastic_rows).set_index("method")
stochastic_summary

## 6) Visualize and interpret the score spread

The methods use different units, so they are shown on separate axes. The dashed line is the deterministic score calculated from the nominal CFs.

In [ ]:
method_colors = {
    "GWP with uncertainty": "#c44e52",
    "Water withdrawal with uncertainty": "#4c72b0",
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ax, label in zip(axes, method_paths):
    scores = stochastic_results[label]["scores"]
    ax.hist(scores, bins=30, color=method_colors[label], alpha=0.85)
    ax.axvline(
        deterministic_results[label]["lca"].score,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Deterministic score",
    )
    ax.set_title(label)
    ax.set_xlabel(method_units[label])
    ax.set_ylabel("Iterations")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Guided checkpoint

Raw histogram widths cannot be compared across different units. Calculate a unitless relative interval width:

\[
\frac{p_{95} - p_{05}}{|p_{50}|}
\]

A larger value means a wider central uncertainty interval relative to the typical score.

In [ ]:
relative_uncertainty = stochastic_summary[[
    "coefficient of variation", "p05", "p50", "p95"
]].copy()
relative_uncertainty["90% interval / median"] = (
    (relative_uncertainty["p95"] - relative_uncertainty["p05"])
    / relative_uncertainty["p50"].abs()
)

relative_uncertainty[[
    "coefficient of variation", "90% interval / median"
]]

## 7) Repeat the workflow with `AWARE 2.0`

The teaching methods use only a few characterization rules. AWARE is a real regionalised method with many location-specific rules and several uncertainty families.

We will:

1. inspect its uncertainty data;
2. calculate a deterministic reference;
3. run a CF-only Monte Carlo;
4. verify that withdrawal and return CFs are sampled symmetrically.

### 7.1 Inspect the packaged uncertainty data

Load the AWARE JSON distributed with `edges`, then create one tidy row per uncertain characterization rule.

In [ ]:
aware_json_path = (
    Path(edges.__file__).resolve().parent
    / "data"
    / "AWARE 2.0_Country_all_yearly.json"
)
aware_method_data = json.loads(aware_json_path.read_text())

aware_uncertainty_rows = []
for rule in aware_method_data["exchanges"]:
    uncertainty = rule.get("uncertainty")
    if uncertainty is None:
        continue

    aware_uncertainty_rows.append({
        "flow": rule["supplier"]["name"],
        "location": str(rule["consumer"]["location"]),
        "nominal CF": rule["value"],
        "distribution": uncertainty["distribution"],
    })

aware_uncertainty_df = pd.DataFrame(aware_uncertainty_rows)
print(
    f'{len(aware_uncertainty_df):,} of '
    f'{len(aware_method_data["exchanges"]):,} AWARE rules contain uncertainty data.'
)

Count the uncertainty families. This is more informative than printing thousands of JSON entries.

In [ ]:
distribution_order = [
    "discrete_empirical",
    "lognorm",
    "triang",
    "weibull_min",
]
distribution_colors = {
    "discrete_empirical": "#4c72b0",
    "lognorm": "#55a868",
    "triang": "#c44e52",
    "weibull_min": "#8172b3",
}
distribution_counts = (
    aware_uncertainty_df["distribution"]
    .value_counts()
    .reindex(distribution_order)
)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(
    distribution_counts.index,
    distribution_counts.values,
    color=[distribution_colors[name] for name in distribution_counts.index],
)
ax.set_title("Uncertainty families in AWARE 2.0")
ax.set_ylabel("Number of uncertain CF rules")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

The consumer location in each rule can be a country code or a more detailed regional code. For the map, keep the country prefix and convert two-letter codes to ISO3.

In [ ]:
country_converter = coco.CountryConverter()
country_codes = country_converter.data[["ISO2", "ISO3"]].dropna().drop_duplicates()
iso2_to_iso3 = country_codes.set_index("ISO2")["ISO3"].astype(str).to_dict()
valid_iso3 = set(country_codes["ISO3"].astype(str))


def location_to_iso3(location):
    prefix = str(location).split("-", 1)[0]

    if len(prefix) == 2:
        return iso2_to_iso3.get(prefix)
    if len(prefix) == 3 and prefix in valid_iso3:
        return prefix
    return None

In [ ]:
aware_uncertainty_df["country iso3"] = (
    aware_uncertainty_df["location"].map(location_to_iso3)
)
aware_country_counts = (
    aware_uncertainty_df.dropna(subset=["country iso3"])
    .groupby("country iso3")
    .size()
    .rename("uncertain CF rules")
    .reset_index()
)

aware_map = go.Figure(go.Choropleth(
    locations=aware_country_counts["country iso3"],
    z=aware_country_counts["uncertain CF rules"],
    colorscale="YlOrRd",
    marker_line_color="white",
    marker_line_width=0.25,
    colorbar_title="Uncertain<br>CF rules",
    hovertemplate="%{location}: %{z} rules<extra></extra>",
))
aware_map.update_geos(
    showcoastlines=False,
    showcountries=True,
    showframe=False,
    projection_type="natural earth",
)
aware_map.update_layout(
    title="Geographic coverage of uncertain AWARE rules",
    height=450,
    margin=dict(t=60, l=0, r=0, b=0),
)
aware_map

Next select one `Water, river` rule for each uncertainty family. We draw 5,000 CF samples from each rule so the distribution shapes can be compared visually.

The dashed line is the nominal CF stored in the method file.

In [ ]:
water_river_examples = {}
water_river_samples = {}

for distribution in distribution_order:
    example = next(
        rule
        for rule in aware_method_data["exchanges"]
        if rule["supplier"]["name"] == "Water, river"
        and rule.get("uncertainty", {}).get("distribution") == distribution
    )
    water_river_examples[distribution] = example
    water_river_samples[distribution] = sample_cf_distribution(
        cf=example,
        n=5_000,
        parameters={},
        random_state=np.random.default_rng(42),
        use_distributions=True,
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, distribution in zip(axes.flat, distribution_order):
    example = water_river_examples[distribution]
    samples = water_river_samples[distribution]
    location = example["consumer"]["location"]

    ax.hist(
        samples,
        bins=35,
        density=True,
        color=distribution_colors[distribution],
        alpha=0.85,
    )
    ax.axvline(
        example["value"],
        color="black",
        linestyle="--",
        label="Nominal CF",
    )
    ax.set_title(f"{distribution}\nWater, river in {location}")
    ax.set_xlabel("Sampled CF")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### 7.2 Calculate deterministic AWARE results

For packaged regionalised methods, `apply_strategies()` runs the geographic matching strategies needed after the inventory is built.

In [ ]:
aware_method = ("AWARE 2.0", "Country", "all", "yearly")

aware_lca = EdgeLCIA(
    demand={activity: 1},
    method=aware_method,
)
aware_lca.apply_strategies()
aware_lca.evaluate_cfs()
aware_lca.lcia()

aware_direct_rows = get_direct_cf_rows(aware_lca, activity)

AWARE assigns opposite-signed CFs to the direct withdrawal and return. Their impacts therefore largely cancel.

The bar chart shows the two direct contributions; the printed values compare their net contribution with the full supply-chain score.

In [ ]:
aware_water_balance = aware_direct_rows[
    aware_direct_rows["supplier name"].isin(["Water", "Water, river"])
].copy()
aware_water_balance["role"] = np.where(
    aware_water_balance["CF"] > 0,
    "Withdrawal",
    "Return to water",
)
aware_water_balance = aware_water_balance.sort_values("impact")

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#55a868" if value < 0 else "#4c72b0" for value in aware_water_balance["impact"]]
ax.barh(aware_water_balance["role"], aware_water_balance["impact"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Direct AWARE water-balance contributions")
ax.set_xlabel("m³ deprived-eq.")
plt.tight_layout()
plt.show()

direct_aware_score = aware_water_balance["impact"].sum()
print(f"Direct net contribution: {direct_aware_score:.2f} m³ deprived-eq.")
print(f"Full supply-chain score: {aware_lca.score:.2f} m³ deprived-eq.")

### 7.3 Run the AWARE Monte Carlo

Now activate the packaged AWARE uncertainty distributions. The inventory remains fixed.

`apply_strategies()` replaces the individual geographic mapping calls and keeps this cell focused on the calculation.

In [ ]:
aware_iterations = 1_000
aware_mc = EdgeLCIA(
    demand={activity: 1},
    method=aware_method,
    use_distributions=True,
    iterations=aware_iterations,
)
aware_mc.lci()
aware_mc.apply_strategies()
aware_mc.evaluate_cfs()
aware_mc.lcia()

aware_scores = np.asarray(aware_mc.score, dtype=float)

In [ ]:
aware_summary = pd.DataFrame([{
    "method": "AWARE 2.0",
    "unit": "m³ deprived-eq.",
    "deterministic score": aware_lca.score,
    **summarize_scores(aware_scores),
}]).set_index("method")

aware_summary

The stochastic characterization table contains many statistics. Keep only the six columns needed to understand the direct water balance.

In [ ]:
aware_mc_direct_rows = get_direct_cf_rows(
    aware_mc,
    activity,
    cf_column="CF (mean)",
)
aware_mc_water_balance = aware_mc_direct_rows[
    aware_mc_direct_rows["supplier name"].isin(["Water", "Water, river"])
].copy()
aware_mc_water_balance["role"] = np.where(
    aware_mc_water_balance["CF (mean)"] > 0,
    "Withdrawal",
    "Return to water",
)

aware_mc_water_balance[[
    "role",
    "supplier name",
    "amount",
    "CF (mean)",
    "CF (std)",
    "impact (mean)",
]]

### 7.4 Check paired withdrawal and return draws

AWARE samples the withdrawal and return CFs as a matched pair: equal magnitude, opposite sign. This correlation is essential—independent draws would create artificial water-balance variability.

The helper below retrieves the per-iteration CF samples for one characterized row. It searches the characterization matrix using the row's flow and consumer metadata.

In [ ]:
def get_cf_samples_for_row(lca, row):
    characterization_matrix = lca.characterization_matrix
    supplier_categories = tuple(row["supplier categories"])

    for supplier_index, consumer_index in zip(
        *characterization_matrix.sum(axis=2).nonzero()
    ):
        supplier = bd.get_activity(lca.reversed_biosphere[supplier_index])
        consumer = bd.get_activity(lca.reversed_activity[consumer_index])

        same_supplier = (
            supplier["name"] == row["supplier name"]
            and tuple(supplier.get("categories", ())) == supplier_categories
        )
        same_consumer = (
            consumer["name"] == row["consumer name"]
            and consumer.get("reference product") == row["consumer reference product"]
            and consumer.get("location") == row["consumer location"]
        )

        if same_supplier and same_consumer:
            samples = characterization_matrix[
                supplier_index, consumer_index, :
            ].todense()
            return np.asarray(samples).ravel().astype(float)

    raise KeyError(f'No characterization samples found for {row["supplier name"]}')

In [ ]:
withdrawal_row = aware_mc_direct_rows.loc[
    aware_mc_direct_rows["CF (mean)"] > 0
].iloc[0]
return_row = aware_mc_direct_rows.loc[
    aware_mc_direct_rows["CF (mean)"] < 0
].iloc[0]

withdrawal_cf_samples = get_cf_samples_for_row(aware_mc, withdrawal_row)
return_cf_samples = get_cf_samples_for_row(aware_mc, return_row)
iterations_to_show = 150

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(
    withdrawal_cf_samples[:iterations_to_show],
    color="#1f77b4",
    label="Withdrawal CF",
)
ax.plot(
    return_cf_samples[:iterations_to_show],
    color="#d62728",
    label="Return CF",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Paired AWARE CF draws")
ax.set_xlabel("Iteration")
ax.set_ylabel("Signed CF")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

pairing_error = np.abs(withdrawal_cf_samples + return_cf_samples).max()
print(f"Maximum pairing error after sign reversal: {pairing_error:.2e}")

Finally, plot the AWARE score distribution. The orange dotted lines show the central 90% interval.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(aware_scores, bins=30, color="#1f77b4", alpha=0.85)
ax.axvline(
    aware_lca.score,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Deterministic score",
)
ax.axvline(
    aware_summary.loc["AWARE 2.0", "mean"],
    color="#d62728",
    linewidth=1.8,
    label="Monte Carlo mean",
)
for quantile in ["p05", "p95"]:
    ax.axvline(
        aware_summary.loc["AWARE 2.0", quantile],
        color="#ff7f0e",
        linestyle=":",
        linewidth=1.8,
        label="5th–95th percentile" if quantile == "p05" else None,
    )

ax.set_title("AWARE 2.0 Monte Carlo: Irrigating (CH)")
ax.set_xlabel("m³ deprived-eq.")
ax.set_ylabel("Iterations")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Interpretation prompt**

- Is the deterministic score close to the Monte Carlo median?
- Is the distribution symmetric?
- Why is preserving the paired withdrawal/return CF draw important for this activity?

## 8) Optional extension: combine inventory and CF uncertainty

So far, only CFs varied. This extension also samples uncertain inventory exchanges.

For a clearer comparison, use the packaged `GLAM3 biodiversity occupation average amphibians` method. It characterizes occupation flows and is less dominated by the irrigation water balance.

### 8.1 Configure the optional comparison

Use 200 iterations to keep the classroom runtime reasonable.

In [ ]:
glam3_method_path = (
    Path(edges.__file__).resolve().parent
    / "data"
    / "GLAM3_biodiversity_occupation_average_amphibians.json"
)
glam3_method = ("teaching", "GLAM3 amphibians occupation")
glam3_method_unit = json.loads(glam3_method_path.read_text())["unit"]
glam3_iterations = 200

### 8.2 Deterministic reference

This is the baseline used in both optional Monte Carlo plots.

In [ ]:
glam3_deterministic = EdgeLCIA(
    demand={activity: 1},
    method=glam3_method,
    filepath=str(glam3_method_path),
)
glam3_deterministic.lci()
glam3_deterministic.apply_strategies()
glam3_deterministic.evaluate_cfs()
glam3_deterministic.lcia()

### 8.3 CF uncertainty only

This run is directly comparable to the Monte Carlo calculations in sections 5 and 7.

In [ ]:
glam3_cf_only = EdgeLCIA(
    demand={activity: 1},
    method=glam3_method,
    filepath=str(glam3_method_path),
    use_distributions=True,
    iterations=glam3_iterations,
)
glam3_cf_only.lci()
glam3_cf_only.apply_strategies()
glam3_cf_only.evaluate_cfs()
glam3_cf_only.lcia()

### 8.4 Inventory and CF uncertainty together

`inventory_use_distributions=True` samples the inventory. `store_inventory_samples=True` keeps exchange-level statistics for later inspection.

Some sampled inventory matrices are poorly conditioned. Repeated solver warnings are suppressed here to keep the optional output readable; a failed solve should still raise an exception.

In [ ]:
glam3_joint = EdgeLCIA(
    demand={activity: 1},
    method=glam3_method,
    filepath=str(glam3_method_path),
    use_distributions=True,
    inventory_use_distributions=True,
    store_inventory_samples=True,
    iterations=glam3_iterations,
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r".*singular matrix.*")
    glam3_joint.lci()
    glam3_joint.apply_strategies()
    glam3_joint.evaluate_cfs()
    glam3_joint.lcia()

### 8.5 Compare the two uncertainty scopes

The same summary function is reused, which makes the comparison consistent with sections 5 and 7.

In [ ]:
glam3_score_sets = {
    "CF uncertainty only": np.asarray(glam3_cf_only.score, dtype=float),
    "Inventory + CF uncertainty": np.asarray(glam3_joint.score, dtype=float),
}

glam3_rows = []
for mode, scores in glam3_score_sets.items():
    glam3_rows.append({
        "mode": mode,
        "unit": glam3_method_unit,
        "iterations": glam3_iterations,
        "deterministic score": glam3_deterministic.score,
        **summarize_scores(scores),
        "negative share": (scores < 0).mean(),
    })

glam3_summary = pd.DataFrame(glam3_rows).set_index("mode")
glam3_summary

Inspect the single direct occupation flow with compact mean and percentile statistics.

In [ ]:
glam3_direct_rows = get_direct_cf_rows(
    glam3_joint,
    activity,
    cf_column="CF (mean)",
)

glam3_direct_rows[[
    "supplier name",
    "supplier categories",
    "amount (mean)",
    "amount (5th)",
    "amount (95th)",
    "CF (mean)",
    "impact (mean)",
]]

In [ ]:
glam3_colors = {
    "CF uncertainty only": "#4c72b0",
    "Inventory + CF uncertainty": "#55a868",
}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

for ax, (mode, scores) in zip(axes, glam3_score_sets.items()):
    ax.hist(scores, bins=25, color=glam3_colors[mode], alpha=0.85)
    ax.axvline(
        glam3_deterministic.score,
        color="black",
        linestyle="--",
        label="Deterministic score",
    )
    ax.axvline(scores.mean(), color="black", label="Mean")
    ax.axvline(
        np.quantile(scores, 0.05),
        color="#ff7f0e",
        linestyle=":",
        linewidth=2,
        label="5th–95th percentile",
    )
    ax.axvline(
        np.quantile(scores, 0.95),
        color="#ff7f0e",
        linestyle=":",
        linewidth=2,
    )
    ax.set_title(mode)
    ax.set_xlabel(glam3_method_unit)
    ax.set_ylabel("Iterations")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
cf_only_width = (
    glam3_summary.loc["CF uncertainty only", "p95"]
    - glam3_summary.loc["CF uncertainty only", "p05"]
)
joint_width = (
    glam3_summary.loc["Inventory + CF uncertainty", "p95"]
    - glam3_summary.loc["Inventory + CF uncertainty", "p05"]
)

print(
    "The joint 90% interval is "
    f"{joint_width / cf_only_width:.1f}× as wide as the CF-only interval."
)

**Interpretation:** if the joint distribution is wider, inventory uncertainty adds relevant variation beyond CF uncertainty. The size of that increase depends on which inventory exchanges the chosen method characterizes.

## Key caution for practitioners

- A deterministic score is a reference, not necessarily the most likely stochastic result.
- A matched-exchange count describes coverage, not importance.
- Always distinguish gross withdrawal from net consumptive use.
- Preserve correlations such as AWARE's paired withdrawal and return CFs.
- Compare uncertainty across units with relative metrics, not raw histogram widths.

## Recap

You have now:

- inspected uncertainty rules in simple and regionalised methods;
- separated direct and upstream contributions;
- run deterministic and CF-only Monte Carlo calculations;
- summarized distributions with quantiles and relative uncertainty metrics;
- verified AWARE's paired withdrawal/return treatment;
- optionally added inventory uncertainty and measured the extra spread.